In [122]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
from datetime import datetime
from urllib.parse import parse_qs
import sqlalchemy
import json
import time
import numpy as np

print('OK')

OK


In [123]:
#url = 'https://okx.com/api/v5/market/history-candles?instId=BTC-USDT&bar=15m&limit=5'
#r = requests.get(url)
#jdata = r.json()['data'][0]

def convert_request_data_to_dict(data, url):
    parsed = requests.utils.urlparse(url)
    query_params = parse_qs(parsed.query)
    asset, interval = query_params.get('instId', [None])[0], query_params.get('bar', [None])[0]
    res_dict = {'asset': asset,
                'interval': interval,
                'timestamp': pd.to_datetime(int(data[0]), utc=False, unit='ms'),
                'open': data[1],
                'high': data[2],
                'low': data[3],
                'close': data[4],
                'volume': data[5]}
    return res_dict


In [124]:
def get_candles(url):
    # url = f'https://okx.com/api/v5/market/history-candles?instId={asset}&after=1704067200000&bar={bar}&limit={limit}'
    r = requests.get(url)
    data = r.json().get('data', [])
    candles = []
    for jdata in data:
        candle = convert_request_data_to_dict(jdata, url)
        candles.append(candle)

    return candles


# get_candles('BTC-USDT', '4H', '300')

# this code only gets first 300 candles per request, but i can get all candles, how?

In [ ]:
def convert_to_timestamp(datetime_str):
    res = datetime.strptime(datetime_str, "%Y-%m-%d %H:%M:%S").timestamp() * 1000
    return res

def convert_dt_for_interval(dt, interval): # currently up to 4H, >= 1d doesnt work
    # '2020-01-12 17:48:39', '15m'
    # returns -> '2020-01-12 17:45:00'
    # how?
    # mb like this?
    # you can use s, m, H
    d, t = dt[:10], dt[11:]
    trim_value, trim_dim = int(interval[:-1]), interval[-1]
    match trim_dim:
        case 's':
           new_s = str((int(t[-2:]) // trim_value) * trim_value)
           t = t[:-2] + new_s
        case 'm':
            new_m = str((int(t[-5:-3]) // trim_value) * trim_value)
            t = t[:-5] + new_m + ':00'
        case 'H':
            new_H = str((int(t[-8:-6]) // trim_value) * trim_value)
            t = t[:-8] + new_H + ':00:00'
    new_dt = d + ' ' + t
    return new_dt


def fetch_candles(asset, bar, limit, after=None):
    base_url = "https://www.okx.com/api/v5/market/history-candles"
    if after is None:
        url = f"{base_url}?instId={asset}&bar={bar}&limit={limit}"
    else:
        url = f"{base_url}?instId={asset}&after={after}&bar={bar}&limit={limit}"
    return get_candles(url)

In [ ]:
def get_okx_server_time():
    r = requests.get("https://www.okx.com/api/v5/public/time")
    return int(r.json()['data'][0]['ts'])

In [ ]:
def fetch_n_candles(asset, bar, n):
    res = []
    batch_limit = 300
    after = None
    while len(res) < n:
        limit = min(batch_limit, n - len(res))
        cc = fetch_candles(asset, bar, limit=limit, after=after)
        if not cc:
            break
        res.extend(cc)
        # oldest candle
        oldest = cc[-1]['timestamp']
        after = int(oldest.timestamp() * 1000)
        time.sleep(0.1) # for test mb can optimize okx allows 20req/2sec
    return res

f = open('res.txt', 'w+')
candles = fetch_n_candles('BTC-USDT', '4H', 10000)
for candle in candles:
    f.write(str(candle) + '\n')